In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import boto3
from io import BytesIO
from itertools import combinations

try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

try:
    import networkx as nx
except:
    ! pip install networkx
    import networkx as nx

## Helper Classes

In [ ]:
class NetworkAnalysis:
    def __init__(self, str_uri, str_bucket, str_dirname_output):
        self.str_uri = str_uri
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_basket = None
        self.graph = None
        self.df_edges = None
        self.dict_communities = None

    def import_data(self):
        self.df_basket = pd.read_parquet(self.str_uri)
        print(f'Basket matrix loaded: {self.df_basket.shape[0]:,} transactions, {self.df_basket.shape[1]:,} items')
        return self

    def build_cooccurrence_graph(self, int_min_cooccurrence=5):
        print(f'\nBuilding co-occurrence graph (min_cooccurrence={int_min_cooccurrence})...')
        # compute pairwise co-occurrences from basket matrix
        arr_basket = self.df_basket.values.astype(int)
        arr_cooccurrence = arr_basket.T @ arr_basket
        np.fill_diagonal(arr_cooccurrence, 0)
        list_items = self.df_basket.columns.tolist()
        # build edge list
        list_edges = []
        for i in range(len(list_items)):
            for j in range(i + 1, len(list_items)):
                int_count = arr_cooccurrence[i, j]
                if int_count >= int_min_cooccurrence:
                    list_edges.append({
                        'item_a': list_items[i],
                        'item_b': list_items[j],
                        'weight': int_count
                    })
        self.df_edges = pd.DataFrame(list_edges)
        # build networkx graph
        self.graph = nx.Graph()
        for _, row in self.df_edges.iterrows():
            self.graph.add_edge(row['item_a'], row['item_b'], weight=row['weight'])
        print(f'Nodes (items): {self.graph.number_of_nodes():,}')
        print(f'Edges (co-occurrences >= {int_min_cooccurrence}): {self.graph.number_of_edges():,}')
        print(f'Graph density: {nx.density(self.graph):.4f}')
        return self

    def compute_centrality(self):
        print('\nComputing centrality metrics...')
        dict_degree = nx.degree_centrality(self.graph)
        dict_betweenness = nx.betweenness_centrality(self.graph, weight='weight')
        dict_weighted_degree = dict(self.graph.degree(weight='weight'))
        self.df_centrality = pd.DataFrame({
            'item': list(dict_degree.keys()),
            'degree_centrality': list(dict_degree.values()),
            'betweenness_centrality': [dict_betweenness[n] for n in dict_degree.keys()],
            'weighted_degree': [dict_weighted_degree[n] for n in dict_degree.keys()]
        }).sort_values('weighted_degree', ascending=False).reset_index(drop=True)
        print(f'\nTop 10 items by weighted degree (total co-occurrences):')
        print(self.df_centrality.head(10).to_string(index=False))
        return self

    def detect_communities(self):
        print('\nDetecting communities (Louvain method)...')
        self.dict_communities = nx.community.louvain_communities(
            self.graph, weight='weight', seed=42
        )
        print(f'Communities found: {len(self.dict_communities)}')
        for idx, set_community in enumerate(self.dict_communities):
            list_sorted = sorted(set_community, 
                                 key=lambda x: dict(self.graph.degree(weight='weight'))[x],
                                 reverse=True)
            str_top_items = ', '.join(list_sorted[:5])
            print(f'  Community {idx+1} ({len(set_community)} items): {str_top_items}...')
        return self

    def plot_network(self):
        fig, ax = plt.subplots(figsize=(14, 14))
        # use top edges for readability
        df_top_edges = self.df_edges.nlargest(100, 'weight')
        graph_sub = nx.Graph()
        for _, row in df_top_edges.iterrows():
            graph_sub.add_edge(row['item_a'], row['item_b'], weight=row['weight'])
        # node sizes by weighted degree
        dict_wd = dict(graph_sub.degree(weight='weight'))
        list_node_sizes = [dict_wd[n] * 0.5 for n in graph_sub.nodes()]
        # edge widths by weight
        list_edge_weights = [graph_sub[u][v]['weight'] for u, v in graph_sub.edges()]
        flt_max_weight = max(list_edge_weights)
        list_edge_widths = [w / flt_max_weight * 4 for w in list_edge_weights]
        # community colors
        dict_node_community = {}
        if self.dict_communities:
            for idx, set_community in enumerate(self.dict_communities):
                for str_node in set_community:
                    dict_node_community[str_node] = idx
        list_colors_palette = plt.cm.Set3(np.linspace(0, 1, max(len(self.dict_communities), 1)))
        list_node_colors = [list_colors_palette[dict_node_community.get(n, 0)] for n in graph_sub.nodes()]
        pos = nx.spring_layout(graph_sub, k=2, seed=42, weight='weight')
        nx.draw_networkx_nodes(graph_sub, pos, node_size=list_node_sizes,
                               node_color=list_node_colors, edgecolors='black',
                               linewidths=0.5, ax=ax)
        nx.draw_networkx_edges(graph_sub, pos, width=list_edge_widths,
                               alpha=0.3, edge_color='gray', ax=ax)
        nx.draw_networkx_labels(graph_sub, pos, font_size=7, ax=ax)
        ax.set_title('Item Co-occurrence Network (Top 100 Edges)', fontsize=14, y=1.02)
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_cooccurrence_network.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_centrality_rankings(self, int_top_n=20):
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        # weighted degree
        df_top = self.df_centrality.head(int_top_n)
        axes[0].barh(range(len(df_top)), df_top['weighted_degree'].values, color='#DD8452', edgecolor='black')
        axes[0].set_yticks(range(len(df_top)))
        axes[0].set_yticklabels(df_top['item'].values, fontsize=9)
        axes[0].set_title(f'Top {int_top_n} Items by Weighted Degree', fontsize=14, y=1.02)
        axes[0].set_xlabel('Weighted Degree (Total Co-occurrences)')
        axes[0].invert_yaxis()
        # betweenness centrality
        df_top_bc = self.df_centrality.nlargest(int_top_n, 'betweenness_centrality')
        axes[1].barh(range(len(df_top_bc)), df_top_bc['betweenness_centrality'].values, color='#DD8452', edgecolor='black')
        axes[1].set_yticks(range(len(df_top_bc)))
        axes[1].set_yticklabels(df_top_bc['item'].values, fontsize=9)
        axes[1].set_title(f'Top {int_top_n} Items by Betweenness Centrality', fontsize=14, y=1.02)
        axes[1].set_xlabel('Betweenness Centrality')
        axes[1].invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_centrality_rankings.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_community_heatmap(self):
        if not self.dict_communities or len(self.dict_communities) < 2:
            print('Not enough communities for heatmap.')
            return self
        # build community-level co-occurrence matrix
        int_n_communities = len(self.dict_communities)
        arr_community_matrix = np.zeros((int_n_communities, int_n_communities))
        dict_node_to_comm = {}
        for idx, set_community in enumerate(self.dict_communities):
            for str_node in set_community:
                dict_node_to_comm[str_node] = idx
        for u, v, d in self.graph.edges(data=True):
            int_c1 = dict_node_to_comm.get(u, -1)
            int_c2 = dict_node_to_comm.get(v, -1)
            if int_c1 >= 0 and int_c2 >= 0:
                arr_community_matrix[int_c1, int_c2] += d['weight']
                if int_c1 != int_c2:
                    arr_community_matrix[int_c2, int_c1] += d['weight']
        list_labels = [f'Community {i+1}\n({len(self.dict_communities[i])} items)' for i in range(int_n_communities)]
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(arr_community_matrix, annot=True, fmt='.0f',
                    xticklabels=list_labels, yticklabels=list_labels,
                    cmap='YlOrRd', ax=ax)
        ax.set_title('Community Co-occurrence Matrix', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_community_heatmap.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_edge_weight_distribution(self):
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(self.df_edges['weight'], bins=50, color='#DD8452', edgecolor='black', alpha=0.8)
        ax.set_title('Edge Weight Distribution (Co-occurrence Counts)', fontsize=14, y=1.02)
        ax.set_xlabel('Co-occurrence Count')
        ax.set_ylabel('Number of Item Pairs')
        ax.axvline(self.df_edges['weight'].median(), color='red', linestyle='--',
                    label=f'Median: {self.df_edges["weight"].median():.0f}')
        ax.set_yscale('log')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_edge_weight_distribution.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def save_results_to_s3(self):
        # save edges
        str_s3_key = '04_network_analysis/edges.parquet'
        print(f'\nSaving edges to S3: s3://{self.str_bucket}/{str_s3_key}')
        buffer = BytesIO()
        self.df_edges.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print('Successfully uploaded edges to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')
        # save centrality
        str_s3_key = '04_network_analysis/centrality.parquet'
        print(f'Saving centrality to S3: s3://{self.str_bucket}/{str_s3_key}')
        buffer = BytesIO()
        self.df_centrality.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print('Successfully uploaded centrality to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')
        return self

    def print_summary(self):
        print('\n=== NETWORK ANALYSIS SUMMARY ===')
        print(f'Nodes (items): {self.graph.number_of_nodes():,}')
        print(f'Edges: {self.graph.number_of_edges():,}')
        print(f'Density: {nx.density(self.graph):.4f}')
        print(f'Communities: {len(self.dict_communities)}')
        if nx.is_connected(self.graph):
            print(f'Average shortest path: {nx.average_shortest_path_length(self.graph):.2f}')
        else:
            int_largest_cc = max(nx.connected_components(self.graph), key=len)
            print(f'Connected components: {nx.number_connected_components(self.graph)}')
            print(f'Largest component: {len(int_largest_cc)} nodes')
        print(f'Average clustering coefficient: {nx.average_clustering(self.graph, weight="weight"):.4f}')
        print(f'\nTop 5 strongest edges (co-occurrences):')
        for _, row in self.df_edges.nlargest(5, 'weight').iterrows():
            print(f'  {row["item_a"]} — {row["item_b"]}: {row["weight"]:,}')
        return self

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = 'network_analysis'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/02_preprocessing/basket_matrix.parquet'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Network Analysis

In [ ]:
cls_network = NetworkAnalysis(str_uri, str_bucket, str_dirname_output)
cls_network.import_data()

In [ ]:
cls_network.build_cooccurrence_graph(int_min_cooccurrence=5)

In [ ]:
cls_network.compute_centrality()

In [ ]:
cls_network.detect_communities()

In [ ]:
cls_network.plot_network()

In [ ]:
cls_network.plot_centrality_rankings()

In [ ]:
cls_network.plot_community_heatmap()

In [ ]:
cls_network.plot_edge_weight_distribution()

In [ ]:
cls_network.print_summary()

In [ ]:
cls_network.save_results_to_s3()

## Completion

In [ ]:
print('\n=== NETWORK ANALYSIS COMPLETE ===')
print(f'Results saved to: s3://{str_bucket}/04_network_analysis/')